# Notebook 01: Data Import and SQL Preprocessing

**Purpose**: Import the COWMAS MySQL database dump into SQLite, run SQL queries to extract and preprocess drinking session data for pen 121 animals.

**Data source**: COWMAS PLF system — RFID scanner readings at water drinkers  
**Period**: October 11 – November 8, 2024  
**Animals**: 34 bulls in pen 121, feedlot in Kazakhstan

In [1]:
import re
import sqlite3
import pandas as pd
import os

# Paths
PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DIR = os.path.join(PROJECT_DIR, 'data', 'raw')
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'data', 'processed')
DB_PATH = os.path.join(PROJECT_DIR, 'data', 'cowmas.db')
SQL_DUMP_PATH = os.path.join(RAW_DIR, 'dump.sql')
EXCEL_PATH = os.path.join(RAW_DIR, 'animals_roster.xlsx')

print(f"Project directory: {PROJECT_DIR}")
print(f"SQL dump exists: {os.path.exists(SQL_DUMP_PATH)}")
print(f"Excel file exists: {os.path.exists(EXCEL_PATH)}")

Project directory: /Users/dimvsh/Downloads/Dana Thesis/project
SQL dump exists: True
Excel file exists: True


## Step 1: Parse MySQL Dump and Load into SQLite

The data was provided as a MySQL dump. We parse it with Python (regex) and load it into a local SQLite database to enable SQL-based preprocessing.

In [2]:
# Read the SQL dump
with open(SQL_DUMP_PATH, 'r') as f:
    sql_content = f.read()

print(f"SQL dump size: {len(sql_content):,} characters")

# Remove the old database if it exists (fresh import)
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# ---- Create tables (SQLite-compatible schema) ----

cursor.execute('''
CREATE TABLE animals (
    animal_id INTEGER PRIMARY KEY,
    farm_id INTEGER NOT NULL,
    breed TEXT NOT NULL,
    birth_date TEXT NOT NULL,
    tag_rfid TEXT NOT NULL,
    tag_legacy TEXT NOT NULL,
    healthy INTEGER NOT NULL,
    culled INTEGER,
    created_at TEXT,
    gender TEXT NOT NULL,
    pen_id INTEGER
)
''')

cursor.execute('''
CREATE TABLE drinkers (
    drinker_id INTEGER PRIMARY KEY,
    drinker_legacy_id TEXT NOT NULL,
    farm_id INTEGER NOT NULL,
    created_at TEXT
)
''')

cursor.execute('''
CREATE TABLE pens (
    pen_id INTEGER PRIMARY KEY,
    pen_number INTEGER NOT NULL,
    farm_id INTEGER NOT NULL,
    neighbour_pen_id INTEGER,
    drinker_id INTEGER,
    created_at TEXT
)
''')

cursor.execute('''
CREATE TABLE scanner_data (
    id INTEGER PRIMARY KEY,
    tag_id TEXT,
    antenna INTEGER,
    start_timestamp TEXT,
    end_timestamp TEXT,
    device_id TEXT
)
''')

conn.commit()
print("SQLite tables created: animals, drinkers, pens, scanner_data")

SQL dump size: 3,215,297 characters
SQLite tables created: animals, drinkers, pens, scanner_data


In [3]:
# ---- Parse INSERT statements from the MySQL dump and load into SQLite ----

def parse_inserts(sql_text, table_name):
    """Extract all value tuples from INSERT INTO `table_name` VALUES (...) statements."""
    pattern = rf"INSERT INTO `{table_name}` VALUES\s*(.+?);"
    matches = re.findall(pattern, sql_text, re.DOTALL)
    
    all_rows = []
    for match in matches:
        # Parse individual value tuples: (val1, val2, ...), (val1, val2, ...)
        # Handle quoted strings that may contain commas and parentheses
        row_pattern = r"\(([^)]+)\)"
        rows = re.findall(row_pattern, match)
        for row in rows:
            # Split by comma, but respect quoted strings
            values = []
            current = ''
            in_quote = False
            for char in row:
                if char == "'" and not in_quote:
                    in_quote = True
                elif char == "'" and in_quote:
                    in_quote = False
                elif char == ',' and not in_quote:
                    values.append(current.strip().strip("'"))
                    current = ''
                    continue
                current += char
            values.append(current.strip().strip("'"))
            all_rows.append(values)
    
    return all_rows

# Parse each table
animals_rows = parse_inserts(sql_content, 'animals')
drinkers_rows = parse_inserts(sql_content, 'drinkers')
pens_rows = parse_inserts(sql_content, 'pens')
scanner_rows = parse_inserts(sql_content, 'scanner_data')

print(f"Parsed from SQL dump:")
print(f"  animals:      {len(animals_rows)} rows")
print(f"  drinkers:     {len(drinkers_rows)} rows")
print(f"  pens:         {len(pens_rows)} rows")
print(f"  scanner_data: {len(scanner_rows)} rows")

Parsed from SQL dump:
  animals:      34 rows
  drinkers:     20 rows
  pens:         40 rows
  scanner_data: 36337 rows


In [4]:
# ---- Insert parsed data into SQLite ----

# Handle NULL values: replace 'NULL' strings with None
def clean_val(v):
    return None if v == 'NULL' else v

# Animals: 11 columns
cursor.executemany(
    'INSERT INTO animals VALUES (?,?,?,?,?,?,?,?,?,?,?)',
    [[clean_val(v) for v in row] for row in animals_rows]
)

# Drinkers: 4 columns
cursor.executemany(
    'INSERT INTO drinkers VALUES (?,?,?,?)',
    [[clean_val(v) for v in row] for row in drinkers_rows]
)

# Pens: 6 columns
cursor.executemany(
    'INSERT INTO pens VALUES (?,?,?,?,?,?)',
    [[clean_val(v) for v in row] for row in pens_rows]
)

# Scanner data: 6 columns
cursor.executemany(
    'INSERT INTO scanner_data VALUES (?,?,?,?,?,?)',
    [[clean_val(v) for v in row] for row in scanner_rows]
)

conn.commit()

# Verify counts
for table in ['animals', 'drinkers', 'pens', 'scanner_data']:
    count = cursor.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f"  {table}: {count} rows loaded into SQLite")

  animals: 34 rows loaded into SQLite
  drinkers: 20 rows loaded into SQLite
  pens: 40 rows loaded into SQLite
  scanner_data: 36337 rows loaded into SQLite


## Step 2: SQL Preprocessing Queries

Now we use SQL to extract and preprocess the data. The key operations:
1. **JOIN** scanner_data with animals to identify pen 121 animals
2. **Deduplicate** sessions — keep only the final ping per drinking visit
3. **Filter** out incomplete sessions and neighboring pen noise

In [5]:
# ---- Query 1: Explore the database structure ----

# Check animals in pen 121
query_animals = """
SELECT a.animal_id, a.tag_rfid, a.tag_legacy, a.breed, a.birth_date, 
       a.healthy, a.culled, a.pen_id, p.pen_number, p.drinker_id
FROM animals a
JOIN pens p ON a.pen_id = p.pen_id
WHERE p.pen_number = 121
ORDER BY a.animal_id
"""
df_animals_pen121 = pd.read_sql_query(query_animals, conn)
print(f"Animals in pen 121: {len(df_animals_pen121)}")
df_animals_pen121.head(10)

Animals in pen 121: 34


,animal_id,tag_rfid,tag_legacy,breed,birth_date,healthy,culled,pen_id,pen_number,drinker_id
0,208,999999999999000000001101,KZP159703128,Ð‘ÐµÑÐ¿Ð¾Ñ€Ð¾Ð´Ð½Ñ‹Ðµ,2022-04-12,1,0,1,121,1
1,209,999999999999000000001120,KZD160323630,ÐšÐ°Ð·Ð°Ñ…ÑÐºÐ°Ñ Ð±ÐµÐ»Ð¾Ð³Ð¾Ð»Ð¾Ð²Ð°Ñ,2023-02-27,1,0,1,121,1
2,210,999999999999000000001140,KZP159416986,Ð‘ÐµÑÐ¿Ð¾Ñ€Ð¾Ð´Ð½Ñ‹Ðµ,2023-02-10,1,0,1,121,1
3,211,999999999999000000001139,KZL101836676,Ð‘ÐµÑÐ¿Ð¾Ñ€Ð¾Ð´Ð½Ñ‹Ðµ,2023-01-21,1,0,1,121,1
4,212,999999999999000000001145,KZP159378792,Ð‘ÐµÑÐ¿Ð¾Ñ€Ð¾Ð´Ð½Ñ‹Ðµ,2022-06-28,1,0,1,121,1
5,213,999999999999000000001138,KZP159476781,Ð‘ÐµÑÐ¿Ð¾Ñ€Ð¾Ð´Ð½Ñ‹Ðµ,2023-02-25,1,0,1,121,1
6,214,999999999999000000001113,KZS180553242,Ð‘ÐµÑÐ¿Ð¾Ñ€Ð¾Ð´Ð½Ñ‹Ðµ,2023-03-01,1,0,1,121,1
7,215,999999999999000000001129,KZP159476793,Ð‘ÐµÑÐ¿Ð¾Ñ€Ð¾Ð´Ð½Ñ‹Ðµ,2023-02-25,1,0,1,121,1
8,216,999999999999000000001144,KZP159596246,Ð‘ÐµÑÐ¿Ð¾Ñ€Ð¾Ð´Ð½Ñ‹Ðµ,2024-10-13,1,1,1,121,1
9,217,999999999999000000001137,KZL102013988,Ð‘ÐµÑÐ¿Ð¾Ñ€Ð¾Ð´Ð½Ñ‹Ðµ,2023-02-09,1,0,1,121,1


In [6]:
# ---- Query 2: Check raw scanner_data volume per tag ----

query_tag_counts = """
SELECT s.tag_id, COUNT(*) as record_count
FROM scanner_data s
GROUP BY s.tag_id
ORDER BY record_count DESC
"""
df_tag_counts = pd.read_sql_query(query_tag_counts, conn)
print(f"Total unique tags in scanner_data: {len(df_tag_counts)}")
print(f"\nTags with > 100 records (primary pen animals):")
primary = df_tag_counts[df_tag_counts['record_count'] > 100]
print(f"  Count: {len(primary)}")
print(f"  Total records: {primary['record_count'].sum():,}")
print(f"\nTags with <= 100 records (neighboring pen noise):")
noise = df_tag_counts[df_tag_counts['record_count'] <= 100]
print(f"  Count: {len(noise)}")
print(f"  Total records: {noise['record_count'].sum():,}")

Total unique tags in scanner_data: 175

Tags with > 100 records (primary pen animals):
  Count: 30
  Total records: 34,986

Tags with <= 100 records (neighboring pen noise):
  Count: 145
  Total records: 1,351


In [7]:
# ---- Query 3: Deduplicate and clean scanner_data ----
# 
# The scanner sends periodic pings during a drinking session. Multiple records 
# share the same (tag_id, start_timestamp) with progressively later end_timestamps.
# We keep only the FINAL record per session (MAX end_timestamp).
# We also filter out:
#   - Sessions with end_timestamp = '0' (session started but never completed)
#   - Tags with fewer than 100 total records (neighboring pen noise)

query_clean = """
WITH tag_filter AS (
    SELECT tag_id
    FROM scanner_data
    GROUP BY tag_id
    HAVING COUNT(*) > 100
),
deduplicated AS (
    SELECT 
        s.tag_id,
        s.antenna,
        s.start_timestamp,
        MAX(s.end_timestamp) AS end_timestamp,
        s.device_id
    FROM scanner_data s
    INNER JOIN tag_filter tf ON s.tag_id = tf.tag_id
    WHERE s.end_timestamp != '0'
    GROUP BY s.tag_id, s.start_timestamp
)
SELECT * FROM deduplicated
ORDER BY tag_id, start_timestamp
"""

df_clean = pd.read_sql_query(query_clean, conn)
print(f"Records after deduplication and filtering: {len(df_clean):,}")
print(f"Unique animals: {df_clean['tag_id'].nunique()}")
df_clean.head(10)

Records after deduplication and filtering: 34,805
Unique animals: 30


,tag_id,antenna,start_timestamp,end_timestamp,device_id
0,999999999999000000001101,1,17286230581751,1728623433041,d4a7af59e92892fb
1,999999999999000000001101,1,17286300010591,1728630001295,d4a7af59e92892fb
2,999999999999000000001101,1,17286300036131,1728630007074,d4a7af59e92892fb
3,999999999999000000001101,1,17286300093811,1728630010863,d4a7af59e92892fb
4,999999999999000000001101,1,17286300207141,1728630024424,d4a7af59e92892fb
5,999999999999000000001101,1,17286300261731,1728630065269,d4a7af59e92892fb
6,999999999999000000001101,1,17286300677051,1728630115784,d4a7af59e92892fb
7,999999999999000000001101,1,17286301175341,1728630184762,d4a7af59e92892fb
8,999999999999000000001101,1,17286397621141,1728639767011,d4a7af59e92892fb
9,999999999999000000001101,1,17286401846731,1728640184714,d4a7af59e92892fb


## Step 3: Timestamp Conversion and Visit Duration Calculation

The timestamps in the database are stored as epoch milliseconds (13-digit strings). We convert them to proper datetime objects and calculate the duration of each drinking visit.

In [8]:
# ---- Convert timestamps from epoch milliseconds to datetime ----

def epoch_ms_to_datetime(ts_str):
    """Convert epoch millisecond string to datetime. Takes first 13 digits."""
    try:
        ts_ms = int(ts_str[:13])
        return pd.Timestamp(ts_ms, unit='ms')
    except (ValueError, TypeError):
        return pd.NaT

df_clean['start_dt'] = df_clean['start_timestamp'].apply(epoch_ms_to_datetime)
df_clean['end_dt'] = df_clean['end_timestamp'].apply(epoch_ms_to_datetime)

# Calculate visit duration in seconds
df_clean['duration_sec'] = (df_clean['end_dt'] - df_clean['start_dt']).dt.total_seconds()

# Extract date for daily aggregation
df_clean['date'] = df_clean['start_dt'].dt.date

# Extract short tag ID (last 4 digits) for readability
df_clean['tag_short'] = df_clean['tag_id'].str[-4:]

print(f"Date range: {df_clean['date'].min()} to {df_clean['date'].max()}")
print(f"\nDuration statistics (seconds):")
print(df_clean['duration_sec'].describe())
df_clean[['tag_short', 'start_dt', 'end_dt', 'duration_sec', 'date']].head(10)

Date range: 2024-10-11 to 2024-11-08

Duration statistics (seconds):
count    34805.000000
mean        12.284527
std        197.692827
min          0.000000
25%          0.504000
50%          2.125000
75%          7.755000
max      27137.131000
Name: duration_sec, dtype: float64


,tag_short,start_dt,end_dt,duration_sec,date
0,1101,2024-10-11 05:04:18.175,2024-10-11 05:10:33.041,374.866,2024-10-11
1,1101,2024-10-11 07:00:01.059,2024-10-11 07:00:01.295,0.236,2024-10-11
2,1101,2024-10-11 07:00:03.613,2024-10-11 07:00:07.074,3.461,2024-10-11
3,1101,2024-10-11 07:00:09.381,2024-10-11 07:00:10.863,1.482,2024-10-11
4,1101,2024-10-11 07:00:20.714,2024-10-11 07:00:24.424,3.710,2024-10-11
5,1101,2024-10-11 07:00:26.173,2024-10-11 07:01:05.269,39.096,2024-10-11
6,1101,2024-10-11 07:01:07.705,2024-10-11 07:01:55.784,48.079,2024-10-11
7,1101,2024-10-11 07:01:57.534,2024-10-11 07:03:04.762,67.228,2024-10-11
8,1101,2024-10-11 09:42:42.114,2024-10-11 09:42:47.011,4.897,2024-10-11
9,1101,2024-10-11 09:49:44.673,2024-10-11 09:49:44.714,0.041,2024-10-11


In [9]:
# ---- Remove implausible durations ----
# A drinking visit lasting 0 seconds or negative is not valid.
# Extremely long visits (e.g., > 30 minutes = 1800s) likely indicate the animal 
# was standing near the drinker without actually drinking.

before = len(df_clean)

# Remove zero/negative durations
df_clean = df_clean[df_clean['duration_sec'] > 0].copy()
after_zero = len(df_clean)

# Check distribution of long visits before deciding cutoff
print("Duration percentiles (seconds):")
for p in [50, 75, 90, 95, 99, 99.5, 100]:
    val = df_clean['duration_sec'].quantile(p / 100)
    print(f"  {p}th percentile: {val:.1f}s ({val/60:.1f} min)")

# Remove visits longer than 30 minutes (1800 seconds)
df_clean = df_clean[df_clean['duration_sec'] <= 1800].copy()
after_long = len(df_clean)

print(f"\nRows removed:")
print(f"  Zero/negative duration: {before - after_zero}")
print(f"  Duration > 30 minutes:  {after_zero - after_long}")
print(f"  Remaining: {after_long:,} drinking visits")

Duration percentiles (seconds):
  50th percentile: 2.4s (0.0 min)
  75th percentile: 8.5s (0.1 min)
  90th percentile: 28.0s (0.5 min)
  95th percentile: 51.5s (0.9 min)
  99th percentile: 111.3s (1.9 min)
  99.5th percentile: 141.6s (2.4 min)
  100th percentile: 27137.1s (452.3 min)

Rows removed:
  Zero/negative duration: 1969
  Duration > 30 minutes:  11
  Remaining: 32,825 drinking visits


### Partial Day Removal

Inspection of daily data coverage revealed two days where data collection was truncated mid-morning:
- **October 15**: scanner data ends at 07:41 AM (391 visits, 24 animals — vs ~1,200 visits on normal days)
- **November 8**: scanner data ends at 09:30 AM (316 visits, 27 animals)

Including these partial days would artificially depress daily metrics (total duration, visit count) and inflate max_absence values, creating data-collection artifacts that the anomaly detection model would misinterpret as behavioral anomalies. We remove them.

In [10]:
import datetime

# Verify the truncation
daily_coverage = df_clean.groupby('date').agg(
    n_visits=('duration_sec', 'count'),
    n_animals=('tag_short', 'nunique'),
    first_record=('start_dt', 'min'),
    last_record=('start_dt', 'max')
).reset_index()

print("Daily data coverage (flagging days where last record is before noon):")
for _, row in daily_coverage.iterrows():
    last_hour = row['last_record'].hour
    flag = " *** TRUNCATED ***" if last_hour < 12 else ""
    print(f"  {row['date']}: {row['n_visits']:>5} visits, {row['n_animals']:>2} animals, "
          f"last record at {row['last_record'].strftime('%H:%M')}{flag}")

# Remove truncated days
partial_days = [datetime.date(2024, 10, 15), datetime.date(2024, 11, 8)]
before = len(df_clean)
df_clean = df_clean[~df_clean['date'].isin(partial_days)].copy()

print(f"\nRemoved partial collection days: {[str(d) for d in partial_days]}")
print(f"Records removed: {before - len(df_clean)}")
print(f"Remaining: {len(df_clean):,} visits across {df_clean['date'].nunique()} days")

Daily data coverage (flagging days where last record is before noon):
  2024-10-11:  1547 visits, 29 animals, last record at 23:38
  2024-10-12:  2098 visits, 29 animals, last record at 22:56
  2024-10-13:  1332 visits, 29 animals, last record at 22:27
  2024-10-14:   985 visits, 29 animals, last record at 23:10
  2024-10-15:   391 visits, 24 animals, last record at 07:41 *** TRUNCATED ***
  2024-10-16:   693 visits, 29 animals, last record at 22:43
  2024-10-17:  1477 visits, 28 animals, last record at 23:04
  2024-10-18:  1614 visits, 29 animals, last record at 22:00
  2024-10-19:   796 visits, 29 animals, last record at 22:09
  2024-10-20:  1362 visits, 30 animals, last record at 22:06
  2024-10-21:  2202 visits, 30 animals, last record at 22:52
  2024-10-22:  1648 visits, 30 animals, last record at 23:54
  2024-10-23:  1342 visits, 30 animals, last record at 21:12
  2024-10-24:  1008 visits, 28 animals, last record at 23:32
  2024-10-25:  1352 visits, 30 animals, last record at 21:

## Step 4: Build the Animals Registry with Health Labels

Merge the database animal records with the veterinary notes from the Excel roster to create a complete animal registry with known health events.

In [11]:
# ---- Load Excel roster and extract health labels ----

df_roster = pd.read_excel(EXCEL_PATH, header=0)
# Column names are in Russian; map by position
# Columns: №, ИНЖ(tag_legacy), ЧИП(chip), Дата рождения, ВЕС, Дата посл, ПОЛ, Клетка, Порода, Дни на откорме, Дата выбытия, Причина
df_roster.columns = ['num', 'tag_legacy', 'chip', 'birth_date', 'weight', 
                     'last_date', 'sex', 'pen', 'breed', 'days_on_feed',
                     'exit_date', 'reason']

# The 'reason' column contains health notes
# Map tag_legacy to health status
health_labels = {}
for _, row in df_roster.iterrows():
    tag = str(row['tag_legacy']).strip()
    reason = str(row['reason']).strip() if pd.notna(row['reason']) else ''
    if reason:
        health_labels[tag] = reason

print("Animals with health/exit notes:")
for tag, reason in health_labels.items():
    print(f"  {tag}: {reason}")

# Build animals dataframe with health labels
df_animals = pd.read_sql_query("""
    SELECT animal_id, tag_rfid, tag_legacy, breed, birth_date, 
           healthy, culled, pen_id
    FROM animals
""", conn)

# Add short tag for matching with scanner_data
df_animals['tag_short'] = df_animals['tag_rfid'].str[-4:]

# Add health labels from Excel
df_animals['health_note'] = df_animals['tag_legacy'].map(
    lambda x: health_labels.get(x.strip(), '')
)
df_animals['is_sick'] = df_animals['health_note'].apply(
    lambda x: 1 if x and x not in ['', 'Реализован'] else 0
)

print(f"\nTotal animals in database: {len(df_animals)}")
print(f"Animals with health issues: {df_animals['is_sick'].sum()}")
df_animals[df_animals['health_note'] != '']

Animals with health/exit notes:
  KZP159476793: пневмония??
  KZD160029574: тимпания факт
  KZP159596246: Реализован
  KZD159558857: МПК Зооветбрак
  KZL102081994: МПК Зооветбрак
  KZL101562994: в/з

Total animals in database: 34
Animals with health issues: 5


,animal_id,tag_rfid,tag_legacy,breed,birth_date,healthy,culled,pen_id,tag_short,health_note,is_sick
7,215,999999999999000000001129,KZP159476793,Ð‘ÐµÑÐ¿Ð¾Ñ€Ð¾Ð´Ð½Ñ‹Ðµ,2023-02-25,1,0,1,1129,пневмония??,1
8,216,999999999999000000001144,KZP159596246,Ð‘ÐµÑÐ¿Ð¾Ñ€Ð¾Ð´Ð½Ñ‹Ðµ,2024-10-13,1,1,1,1144,Реализован,0
15,223,999999999999000000001118,KZD159558857,Ð‘ÐµÑÐ¿Ð¾Ñ€Ð¾Ð´Ð½Ñ‹Ðµ,2024-10-13,1,1,1,1118,МПК Зооветбрак,1
21,229,999999999999000000001148,KZL102081994,Ð‘ÐµÑÐ¿Ð¾Ñ€Ð¾Ð´Ð½Ñ‹Ðµ,2024-10-13,1,1,1,1148,МПК Зооветбрак,1
22,230,999999999999000000001149,KZD160029574,Ð‘ÐµÑÐ¿Ð¾Ñ€Ð¾Ð´Ð½Ñ‹Ðµ,2023-02-05,1,0,1,1149,тимпания факт,1
32,240,999999999999000000001104,KZL101562994,ÐšÐ°Ð·Ð°Ñ…ÑÐºÐ°Ñ Ð±ÐµÐ»Ð¾Ð³Ð¾Ð»Ð¾Ð²Ð°Ñ,2024-10-13,1,1,1,1104,в/з,1


## Step 5: Final Summary and Export

Save the cleaned data to CSV files for use in subsequent notebooks.

In [12]:
# ---- Final summary ----

print("=" * 60)
print("DATA PREPROCESSING SUMMARY")
print("=" * 60)
print(f"Raw scanner_data records:       {len(scanner_rows):,}")
print(f"After deduplication + cleaning: {len(df_clean):,}")
print(f"Unique animals retained:        {df_clean['tag_short'].nunique()}")
print(f"Date range:                     {df_clean['date'].min()} to {df_clean['date'].max()}")
print(f"Days covered:                   {df_clean['date'].nunique()}")
print(f"")
print(f"Visit duration (seconds):")
print(f"  Mean:   {df_clean['duration_sec'].mean():.1f}")
print(f"  Median: {df_clean['duration_sec'].median():.1f}")
print(f"  Min:    {df_clean['duration_sec'].min():.1f}")
print(f"  Max:    {df_clean['duration_sec'].max():.1f}")
print(f"")

# Per-animal summary
animal_summary = df_clean.groupby('tag_short').agg(
    total_visits=('duration_sec', 'count'),
    total_duration_sec=('duration_sec', 'sum'),
    avg_visit_sec=('duration_sec', 'mean'),
    days_active=('date', 'nunique')
).round(1)
print(f"Per-animal summary:")
print(animal_summary.to_string())

DATA PREPROCESSING SUMMARY
Raw scanner_data records:       36,337
After deduplication + cleaning: 32,118
Unique animals retained:        30
Date range:                     2024-10-11 to 2024-11-07
Days covered:                   27

Visit duration (seconds):
  Mean:   10.2
  Median: 2.4
  Min:    0.0
  Max:    906.9

Per-animal summary:
           total_visits  total_duration_sec  avg_visit_sec  days_active
tag_short                                                              
1101               1328             16842.4           12.7           27
1105               1268             12309.1            9.7           27
1106               1046             16655.6           15.9           27
1110                503              3446.5            6.9           27
1111               1266             11734.7            9.3           27
1112               1162             13950.0           12.0           27
1113               2402             13481.1            5.6           27
1117         

In [13]:
# ---- Export to CSV ----

# Scanner data (cleaned visits)
export_cols = ['tag_id', 'tag_short', 'antenna', 'start_dt', 'end_dt', 
               'duration_sec', 'date', 'device_id']
df_clean[export_cols].to_csv(
    os.path.join(PROCESSED_DIR, 'scanner_data_clean.csv'), index=False
)

# Animals registry
df_animals.to_csv(
    os.path.join(PROCESSED_DIR, 'animals.csv'), index=False
)

print(f"Exported:")
print(f"  scanner_data_clean.csv  ({len(df_clean):,} rows)")
print(f"  animals.csv             ({len(df_animals)} rows)")
print(f"\nFiles saved to: {PROCESSED_DIR}")

# Close database connection
conn.close()
print(f"SQLite database saved to: {DB_PATH}")

Exported:
  scanner_data_clean.csv  (32,118 rows)
  animals.csv             (34 rows)

Files saved to: /Users/dimvsh/Downloads/Dana Thesis/project/data/processed
SQLite database saved to: /Users/dimvsh/Downloads/Dana Thesis/project/data/cowmas.db
